# Path-Level Metrics
## Inconsistency Rate:
- % of multi-step paths where the model flips between correct and incorrect (i.e., inconsistent).
- High rate → suggests fragile generalization or local reasoning failures.

In [3]:
import json
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def compute_inconsistency_rate(data, group_key="difficulty"):
    grouped = defaultdict(list)

    for entry in data:
        path = entry["path"]
        if len(path) <= 1:
            continue  # only multi-step paths

        group_value = path[0].get(group_key)
        correctness = [step["correct"] for step in path]

        has_correct = any(correctness)
        has_incorrect = any(c == 0 for c in correctness)

        if has_correct and has_incorrect:
            grouped[group_value].append("inconsistent")
        else:
            grouped[group_value].append("consistent")

    results = {}
    for group, paths in grouped.items():
        total = len(paths)
        inconsistent = paths.count("inconsistent")
        results[group] = {
            "total_paths": total,
            "inconsistent_paths": inconsistent,
            "inconsistency_rate": round(inconsistent / total, 3) if total > 0 else 0
        }

    return results

# Example: Inconsistency rate by difficulty
inconsistency_by_difficulty = compute_inconsistency_rate(data, group_key="difficulty")

# Print results
print("Inconsistent Paths Percentage (by difficulty)")
for difficulty, stats in inconsistency_by_difficulty.items():
    print(f"{difficulty}: {stats['inconsistency_rate']*100:.1f}% inconsistent "
          f"({stats['inconsistent_paths']}/{stats['total_paths']} paths)")


Inconsistent Paths Percentage (by difficulty)
moderate: 17.8% inconsistent (23/129 paths)
challenging: 8.9% inconsistent (4/45 paths)
simple: 13.6% inconsistent (6/44 paths)


In [2]:
import json
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def compute_inconsistency_rate(data, group_key="difficulty"):
    grouped = defaultdict(list)

    for entry in data:
        path = entry["path"]
        if len(path) <= 1:
            continue  # only multi-step paths

        group_value = path[0].get(group_key)
        correctness = [step["correct"] for step in path]

        has_correct = any(correctness)
        has_incorrect = any(c == 0 for c in correctness)

        if has_correct and has_incorrect:
            grouped[group_value].append("inconsistent")
        else:
            grouped[group_value].append("consistent")

    results = {}
    for group, paths in grouped.items():
        total = len(paths)
        inconsistent = paths.count("inconsistent")
        results[group] = {
            "total_paths": total,
            "inconsistent_paths": inconsistent,
            "inconsistency_rate": round(inconsistent / total, 3) if total > 0 else 0
        }

    return results

# Example: Inconsistency rate by difficulty
inconsistency_by_difficulty = compute_inconsistency_rate(data, group_key="db_id")

# Print results
print("Inconsistent Paths Percentage (by db_id)")
for difficulty, stats in inconsistency_by_difficulty.items():
    print(f"{difficulty}: {stats['inconsistency_rate']*100:.1f}% inconsistent "
          f"({stats['inconsistent_paths']}/{stats['total_paths']} paths)")


Inconsistent Paths Percentage (by db_id)
debit_card_specializing: 16.7% inconsistent (2/12 paths)
student_club: 25.0% inconsistent (5/20 paths)
thrombosis_prediction: 0.0% inconsistent (0/28 paths)
european_football_2: 9.5% inconsistent (2/21 paths)
formula_1: 27.0% inconsistent (10/37 paths)
superhero: 13.3% inconsistent (2/15 paths)
codebase_community: 33.3% inconsistent (3/9 paths)
card_games: 25.0% inconsistent (7/28 paths)
toxicology: 0.0% inconsistent (0/11 paths)
california_schools: 12.5% inconsistent (2/16 paths)
financial: 0.0% inconsistent (0/21 paths)


## Consistency Rate:
- % of multi-step paths where the model is consistent through the whole path (either always correct or always incorrect)
- High rate (correct) → suggests strong ability in this category.
- High rate (incorrect) → suggests a strong weakness in this category.

Note:
Which kind of errors could be correlated
Which type of error oscelates the most
Which erros tend to be solved and remain solved and which tend to move around

In [14]:
import json
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def compute_consistent_path_stats(data, group_key="difficulty"):
    stats = defaultdict(lambda: {"all_correct": 0, "all_incorrect": 0, "total_consistent": 0})

    for entry in data:
        path = entry["path"]
        if len(path) <= 1:
            continue

        correctness = [step["correct"] for step in path]
        group_value = path[0].get(group_key)

        if all(correctness):
            stats[group_value]["all_correct"] += 1
            stats[group_value]["total_consistent"] += 1
        elif not any(correctness):
            stats[group_value]["all_incorrect"] += 1
            stats[group_value]["total_consistent"] += 1

    results = {}
    for group, group_stats in stats.items():
        results[group] = {
            "total_consistent_paths": group_stats["total_consistent"],
            "all_correct": group_stats["all_correct"],
            "all_incorrect": group_stats["all_incorrect"],
            "all_correct_rate": round(group_stats["all_correct"] / group_stats["total_consistent"], 3)
                if group_stats["total_consistent"] > 0 else None,
            "all_incorrect_rate": round(group_stats["all_incorrect"] / group_stats["total_consistent"], 3)
                if group_stats["total_consistent"] > 0 else None,
        }

    return results

# Example: Compute for difficulty
consistent_path_stats = compute_consistent_path_stats(data, group_key="difficulty")

# Print summary
print("Consistent Paths Percentage (by difficulty)")
for diff, stats in consistent_path_stats.items():
    print(f"{diff} — Total: {stats['total_consistent_paths']}, "
          f"All Correct: {stats['all_correct']} ({stats['all_correct_rate']}), "
          f"All Incorrect: {stats['all_incorrect']} ({stats['all_incorrect_rate']})")


Consistent Paths Percentage (by difficulty)
moderate — Total: 102, All Correct: 30 (0.294), All Incorrect: 72 (0.706)
challenging — Total: 41, All Correct: 5 (0.122), All Incorrect: 36 (0.878)
simple — Total: 35, All Correct: 20 (0.571), All Incorrect: 15 (0.429)


In [12]:
import json
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def compute_consistent_path_stats(data, group_key="difficulty"):
    stats = defaultdict(lambda: {"all_correct": 0, "all_incorrect": 0, "total_consistent": 0})

    for entry in data:
        path = entry["path"]
        if len(path) <= 1:
            continue

        correctness = [step["correct"] for step in path]
        group_value = path[0].get(group_key)

        if all(correctness):
            stats[group_value]["all_correct"] += 1
            stats[group_value]["total_consistent"] += 1
        elif not any(correctness):
            stats[group_value]["all_incorrect"] += 1
            stats[group_value]["total_consistent"] += 1

    results = {}
    for group, group_stats in stats.items():
        results[group] = {
            "total_consistent_paths": group_stats["total_consistent"],
            "all_correct": group_stats["all_correct"],
            "all_incorrect": group_stats["all_incorrect"],
            "all_correct_rate": round(group_stats["all_correct"] / group_stats["total_consistent"], 3)
                if group_stats["total_consistent"] > 0 else None,
            "all_incorrect_rate": round(group_stats["all_incorrect"] / group_stats["total_consistent"], 3)
                if group_stats["total_consistent"] > 0 else None,
        }

    return results

# Example: Compute for difficulty
consistent_path_stats = compute_consistent_path_stats(data, group_key="db_id")

# Print summary
print("Consistent Paths Percentage (by db_id)")
for diff, stats in consistent_path_stats.items():
    print(f"{diff} — Total: {stats['total_consistent_paths']}, "
          f"All Correct: {stats['all_correct']} ({stats['all_correct_rate']}), "
          f"All Incorrect: {stats['all_incorrect']} ({stats['all_incorrect_rate']})")


Consistent Paths Percentage (by db_id)
debit_card_specializing — Total: 10, All Correct: 4 (0.4), All Incorrect: 6 (0.6)
student_club — Total: 14, All Correct: 10 (0.714), All Incorrect: 4 (0.286)
thrombosis_prediction — Total: 27, All Correct: 0 (0.0), All Incorrect: 27 (1.0)
european_football_2 — Total: 18, All Correct: 5 (0.278), All Incorrect: 13 (0.722)
formula_1 — Total: 25, All Correct: 6 (0.24), All Incorrect: 19 (0.76)
superhero — Total: 13, All Correct: 10 (0.769), All Incorrect: 3 (0.231)
codebase_community — Total: 6, All Correct: 1 (0.167), All Incorrect: 5 (0.833)
card_games — Total: 21, All Correct: 8 (0.381), All Incorrect: 13 (0.619)
toxicology — Total: 11, All Correct: 2 (0.182), All Incorrect: 9 (0.818)
california_schools — Total: 13, All Correct: 1 (0.077), All Incorrect: 12 (0.923)
financial — Total: 20, All Correct: 8 (0.4), All Incorrect: 12 (0.6)


## Query Length vs Accuracy:
- Track how query complexity (e.g. token count, number of joins/subqueries) correlates with correctness.
- Long queries may correlate with higher hallucination.

In [25]:
import json
import re
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def count_joins_subqueries(query):
    join_count = len(re.findall(r'\bjoin\b', query, re.IGNORECASE))
    subquery_count = len(re.findall(r'\(\s*select\b', query, re.IGNORECASE))
    return join_count + subquery_count

def compute_query_length_vs_accuracy(data, group_key="db_id"):
    grouped_stats = defaultdict(lambda: {
        "total_queries": 0,
        "correct_queries": 0,
        "token_counts": [],
        "complexities": []
    })

    for entry in data:
        path = entry.get("path", [])
        if not path:
            continue

        group_value = path[0].get(group_key, "unknown")

        for step in path:
            predicted_query = step.get("predicted_query", "")
            correct = step.get("correct", 0)
            if correct == None:
                correct = 0

            if not predicted_query:
                continue

            tokens = re.findall(r'\b\w+\b', predicted_query)
            token_count = len(tokens)
            complexity = count_joins_subqueries(predicted_query)

            stats = grouped_stats[group_value]
            stats["total_queries"] += 1
            stats["correct_queries"] += correct
            stats["token_counts"].append(token_count)
            stats["complexities"].append(complexity)

    results = {}
    for group, stats in grouped_stats.items():
        total = stats["total_queries"]
        correct = stats["correct_queries"]
        avg_token_count = sum(stats["token_counts"]) / total if total > 0 else 0
        avg_complexity = sum(stats["complexities"]) / total if total > 0 else 0
        accuracy = correct / total if total > 0 else 0

        results[group] = {
            "total_queries": total,
            "accuracy": round(accuracy, 3),
            "avg_token_count": round(avg_token_count, 1),
            "avg_complexity": round(avg_complexity, 2)
        }

    return results

# Example usage: Group by 'db_id'
query_length_vs_accuracy_by_db = compute_query_length_vs_accuracy(data, group_key="difficulty")

print("Query Length vs Accuracy (by difficulty):")
for db_id, stats in query_length_vs_accuracy_by_db.items():
    print(f"{db_id}: Accuracy={stats['accuracy']} | Avg Tokens={stats['avg_token_count']} | Avg Joins+Subqueries={stats['avg_complexity']} | Total Queries={stats['total_queries']}")


Query Length vs Accuracy (by difficulty):
simple: Accuracy=0.505 | Avg Tokens=18.2 | Avg Joins+Subqueries=0.71 | Total Queries=198
moderate: Accuracy=0.302 | Avg Tokens=26.0 | Avg Joins+Subqueries=1.14 | Total Queries=417
challenging: Accuracy=0.139 | Avg Tokens=35.8 | Avg Joins+Subqueries=1.48 | Total Queries=166


In [24]:
import json
import re
from collections import defaultdict

# Load your data
with open("/Users/hmmoore/Desktop/mini_dev/llm/exp_result/exp_22/predict_mini_dev_gpt-4-turbo_cot_PostgreSQL_cleaned_queries_log.txt", "r") as f:
    data = json.load(f)

def count_joins_subqueries(query):
    join_count = len(re.findall(r'\bjoin\b', query, re.IGNORECASE))
    subquery_count = len(re.findall(r'\(\s*select\b', query, re.IGNORECASE))
    return join_count + subquery_count

def compute_query_length_vs_accuracy(data, group_key="db_id"):
    grouped_stats = defaultdict(lambda: {
        "total_queries": 0,
        "correct_queries": 0,
        "token_counts": [],
        "complexities": []
    })

    for entry in data:
        path = entry.get("path", [])
        if not path:
            continue

        group_value = path[0].get(group_key, "unknown")

        for step in path:
            predicted_query = step.get("predicted_query", "")
            correct = step.get("correct", 0)
            if correct == None:
                correct = 0

            if not predicted_query:
                continue

            tokens = re.findall(r'\b\w+\b', predicted_query)
            token_count = len(tokens)
            complexity = count_joins_subqueries(predicted_query)

            stats = grouped_stats[group_value]
            stats["total_queries"] += 1
            stats["correct_queries"] += correct
            stats["token_counts"].append(token_count)
            stats["complexities"].append(complexity)

    results = {}
    for group, stats in grouped_stats.items():
        total = stats["total_queries"]
        correct = stats["correct_queries"]
        avg_token_count = sum(stats["token_counts"]) / total if total > 0 else 0
        avg_complexity = sum(stats["complexities"]) / total if total > 0 else 0
        accuracy = correct / total if total > 0 else 0

        results[group] = {
            "total_queries": total,
            "accuracy": round(accuracy, 3),
            "avg_token_count": round(avg_token_count, 1),
            "avg_complexity": round(avg_complexity, 2)
        }

    return results

# Example usage: Group by 'db_id'
query_length_vs_accuracy_by_db = compute_query_length_vs_accuracy(data, group_key="db_id")

print("Query Length vs Accuracy (by db_id):")
for db_id, stats in query_length_vs_accuracy_by_db.items():
    print(f"{db_id}: Accuracy={stats['accuracy']} | Avg Tokens={stats['avg_token_count']} | Avg Joins+Subqueries={stats['avg_complexity']} | Total Queries={stats['total_queries']}")


Query Length vs Accuracy (by db_id):
debit_card_specializing: Accuracy=0.333 | Avg Tokens=29.2 | Avg Joins+Subqueries=0.89 | Total Queries=45
student_club: Accuracy=0.597 | Avg Tokens=23.1 | Avg Joins+Subqueries=1.07 | Total Queries=72
thrombosis_prediction: Accuracy=0.011 | Avg Tokens=28.9 | Avg Joins+Subqueries=1.13 | Total Queries=91
european_football_2: Accuracy=0.203 | Avg Tokens=21.6 | Avg Joins+Subqueries=0.93 | Total Queries=74
formula_1: Accuracy=0.317 | Avg Tokens=28.4 | Avg Joins+Subqueries=1.34 | Total Queries=120
superhero: Accuracy=0.662 | Avg Tokens=27.0 | Avg Joins+Subqueries=1.62 | Total Queries=68
codebase_community: Accuracy=0.379 | Avg Tokens=25.5 | Avg Joins+Subqueries=1.0 | Total Queries=58
card_games: Accuracy=0.371 | Avg Tokens=20.7 | Avg Joins+Subqueries=0.72 | Total Queries=89
toxicology: Accuracy=0.245 | Avg Tokens=26.9 | Avg Joins+Subqueries=1.23 | Total Queries=53
california_schools: Accuracy=0.111 | Avg Tokens=27.1 | Avg Joins+Subqueries=0.69 | Total Queri